# Final Algorithm Comparison

This notebook compares the final MDGP local-search heuristic with the selected comparison algorithms.

The following algorithms are evaluated:

- `kapoce`
- `leiden`
- `leiden_mdgp`
- `mdgp_plateau`

The analysis compares

1. relative solution quality,
3. runtime,

All analyses are performed separately by graph type, size class, and density regime.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [2]:
GRAPH_ORDER = [
    "powerlaw",
    "er",
]

DATASET_ORDER = [
    "small sparse",
    "small dense",
    "large sparse",
    "large dense",
]

ALGORITHM_ORDER = [
    "leiden",
    "leiden_mdgp",
    "kapoce",
    "mdgp_plateau",
]

REFERENCE_ALGORITHM = "mdgp_plateau"

RESULTS_DIR = Path("../../results/experiment3")
RAW_RESULTS_FILE = RESULTS_DIR / "raw_results.csv"

## Load and verify experiment data

Before the analysis, the notebook verifies that all expected algorithms and dataset groups are contained in the result file.

For every graph instance, exactly one result per algorithm is expected.

In [3]:
raw = pd.read_csv(RAW_RESULTS_FILE)

required_columns = {
    "dataset",
    "size_class",
    "graph_type",
    "regime",
    "instance",
    "n",
    "m",
    "edge_density",
    "algorithm",
    "density",
    "num_clusters",
    "max_cluster_size",
    "avg_cluster_size",
    "runtime",
}

missing_columns = required_columns.difference(raw.columns)

if missing_columns:
    raise ValueError("Missing required columns: " + ", ".join(sorted(missing_columns)))

raw["dataset_group"] = (
        raw["size_class"].astype(str)
        + " "
        + raw["regime"].astype(str)
)

raw["graph_type"] = pd.Categorical(
    raw["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

raw["dataset_group"] = pd.Categorical(
    raw["dataset_group"],
    categories=DATASET_ORDER,
    ordered=True,
)

raw["algorithm"] = pd.Categorical(
    raw["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

print(f"Loaded {len(raw):,} rows from {RAW_RESULTS_FILE}")

experiment_check = (
    raw
    .groupby(
        ["graph_type", "dataset_group", "algorithm"],
        observed=True,
        as_index=False,
    )
    .agg(
        num_instances=("instance", "nunique"),
        num_rows=("instance", "size"),
    )
    .sort_values(["graph_type", "dataset_group", "algorithm"])
    .reset_index(drop=True)
)

experiment_check

Loaded 8,000 rows from ../../results/experiment3/raw_results.csv


,graph_type,dataset_group,algorithm,num_instances,num_rows
0,powerlaw,small sparse,leiden,250,250
1,powerlaw,small sparse,leiden_mdgp,250,250
2,powerlaw,small sparse,kapoce,250,250
3,powerlaw,small sparse,mdgp_plateau,250,250
4,powerlaw,small dense,leiden,250,250
5,powerlaw,small dense,leiden_mdgp,250,250
6,powerlaw,small dense,kapoce,250,250
7,powerlaw,small dense,mdgp_plateau,250,250
8,powerlaw,large sparse,leiden,250,250
9,powerlaw,large sparse,leiden_mdgp,250,250


## Relative solution quality

For every instance, the best solution found by any of the compared algorithms is used as the reference.

Relative solution quality is defined as

$
\frac{\text{best solution quality on the instance}}
{\text{solution quality of the respective algorithm}}.
$

A value of $1.0$ indicates that the algorithm attains the best solution found on the instance. Values greater than $1.0$ indicate the remaining quality gap.

In [4]:
instance_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
]

density_table = (
    raw
    .pivot(
        index=instance_keys,
        columns="algorithm",
        values="density",
    )
)

density_table = density_table[ALGORITHM_ORDER]
density_table.columns.name = "algorithm"

if density_table.isna().any().any():
    incomplete_instances = density_table[density_table.isna().any(axis=1)]

    raise ValueError(f"{len(incomplete_instances)} instances do not contain density results for every algorithm.")

In [5]:
best_per_instance = density_table.max(axis=1)

relative_to_best = density_table.rdiv( best_per_instance, axis=0)

relative_to_best.columns.name = "algorithm"

relative_quality_summary = (
    relative_to_best
    .groupby(level=["graph_type", "dataset_group"])
    .agg(["mean", "min", "max"])
    .stack(level=0, future_stack=True)
    .reset_index()
    .rename(
        columns={
            "mean": "mean_relative_to_best",
            "min": "min_relative_to_best",
            "max": "max_relative_to_best",
        }
    )
)

relative_quality_summary["algorithm"] = pd.Categorical(
    relative_quality_summary["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

relative_quality_summary = (
    relative_quality_summary
    .sort_values(["graph_type", "dataset_group", "algorithm"])
    .reset_index(drop=True)
)

relative_quality_summary

,graph_type,dataset_group,algorithm,mean_relative_to_best,min_relative_to_best,max_relative_to_best
0,powerlaw,small sparse,leiden,2.970392,1.483051,4.515369
1,powerlaw,small sparse,leiden_mdgp,1.095257,1.028912,1.215663
2,powerlaw,small sparse,kapoce,1.049708,1.013575,1.122106
3,powerlaw,small sparse,mdgp_plateau,1.000000,1.000000,1.000000
4,powerlaw,small dense,leiden,2.538343,1.257303,4.191782
5,powerlaw,small dense,leiden_mdgp,1.132796,1.028493,1.190869
6,powerlaw,small dense,kapoce,1.047734,1.008005,1.139047
7,powerlaw,small dense,mdgp_plateau,1.000000,1.000000,1.000000
8,powerlaw,large sparse,leiden,9.873555,6.236999,14.036937
9,powerlaw,large sparse,leiden_mdgp,1.098397,1.075767,1.121990


## Winner rate

An algorithm is counted as a winner on an instance if it attains the highest solution quality found by any compared algorithm.

Ties are counted as wins for all algorithms attaining the best value.

In [6]:
winner_mask = density_table.eq(best_per_instance,axis=0)

winner_mask.columns.name = "algorithm"

winner_summary = (
    winner_mask
    .groupby(level=["graph_type", "dataset_group",])
    .agg(["sum", "mean"])
    .stack(level=0, future_stack=True)
    .reset_index()
    .rename(
        columns={
            "sum": "best_count_including_ties",
            "mean": "winner_rate",
        }
    )
)

winner_summary["algorithm"] = pd.Categorical(
    winner_summary["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

winner_summary = (
    winner_summary
    .sort_values(["graph_type", "dataset_group", "algorithm"])
    .reset_index(drop=True)
)

winner_summary

,graph_type,dataset_group,algorithm,best_count_including_ties,winner_rate
0,powerlaw,small sparse,leiden,0,0.000
1,powerlaw,small sparse,leiden_mdgp,0,0.000
2,powerlaw,small sparse,kapoce,0,0.000
3,powerlaw,small sparse,mdgp_plateau,250,1.000
4,powerlaw,small dense,leiden,0,0.000
5,powerlaw,small dense,leiden_mdgp,0,0.000
6,powerlaw,small dense,kapoce,0,0.000
7,powerlaw,small dense,mdgp_plateau,250,1.000
8,powerlaw,large sparse,leiden,0,0.000
9,powerlaw,large sparse,leiden_mdgp,0,0.000


## Runtime

The runtime comparison uses the complete runtime of each partitioning algorithm.

For every algorithm, the arithmetic mean of the runtimes is computed over all instances of the corresponding graph and dataset group.

In [7]:
runtime_summary = (
    raw
    .groupby(
        ["graph_type", "dataset_group", "algorithm"],
        observed=True,
        as_index=False,
    )
    .agg(
        mean_runtime=("runtime", "mean"),
        min_runtime=("runtime", "min"),
        max_runtime=("runtime", "max"),
    )
)

runtime_summary["algorithm"] = pd.Categorical(
    runtime_summary["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

runtime_summary = (
    runtime_summary
    .sort_values(["graph_type", "dataset_group", "algorithm"])
    .reset_index(drop=True)
)

runtime_summary

,graph_type,dataset_group,algorithm,mean_runtime,min_runtime,max_runtime
0,powerlaw,small sparse,leiden,0.053121,0.006420,0.173480
1,powerlaw,small sparse,leiden_mdgp,0.004071,0.001070,0.020319
2,powerlaw,small sparse,kapoce,0.142645,0.009298,0.623888
3,powerlaw,small sparse,mdgp_plateau,30.379303,6.762418,80.422641
4,powerlaw,small dense,leiden,0.015139,0.001621,0.071700
5,powerlaw,small dense,leiden_mdgp,0.003337,0.000625,0.018787
6,powerlaw,small dense,kapoce,0.041539,0.007423,0.126750
7,powerlaw,small dense,mdgp_plateau,52.823186,7.095227,125.135710
8,powerlaw,large sparse,leiden,0.107400,0.016991,0.396745
9,powerlaw,large sparse,leiden_mdgp,0.015293,0.003709,0.083077


In [8]:
comparison_summary = (
    relative_quality_summary[["graph_type", "dataset_group", "algorithm", "mean_relative_to_best"]]
    .merge(
        winner_summary[["graph_type", "dataset_group", "algorithm", "winner_rate"]],
        on=["graph_type", "dataset_group", "algorithm"],
        validate="one_to_one",
    )
    .merge(
        runtime_summary[["graph_type", "dataset_group", "algorithm", "mean_runtime",]],
        on=["graph_type", "dataset_group", "algorithm"],
        validate="one_to_one",
    )
)

comparison_summary["algorithm"] = pd.Categorical(
    comparison_summary["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

comparison_summary = (
    comparison_summary
    .sort_values(["graph_type", "dataset_group", "algorithm"])
    .reset_index(drop=True)
)

comparison_summary

,graph_type,dataset_group,algorithm,mean_relative_to_best,winner_rate,mean_runtime
0,powerlaw,small sparse,leiden,2.970392,0.000,0.053121
1,powerlaw,small sparse,leiden_mdgp,1.095257,0.000,0.004071
2,powerlaw,small sparse,kapoce,1.049708,0.000,0.142645
3,powerlaw,small sparse,mdgp_plateau,1.000000,1.000,30.379303
4,powerlaw,small dense,leiden,2.538343,0.000,0.015139
5,powerlaw,small dense,leiden_mdgp,1.132796,0.000,0.003337
6,powerlaw,small dense,kapoce,1.047734,0.000,0.041539
7,powerlaw,small dense,mdgp_plateau,1.000000,1.000,52.823186
8,powerlaw,large sparse,leiden,9.873555,0.000,0.107400
9,powerlaw,large sparse,leiden_mdgp,1.098397,0.000,0.015293


## LaTeX helper functions

The following functions format algorithm names and numerical values for the thesis tables.

In [9]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_algorithm(algorithm: str) -> str:
    return r"\texttt{" + algorithm.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"


def format_percent(value: float, decimals: int) -> str:
    return f"{truncate_number(100 * value, decimals):.{decimals}f}" + r"\,\%"

## Build LaTeX comparison tables

The tables report mean relative solution quality, winner rate, and mean runtime.

Within each dataset group, the best mean relative solution quality is highlighted in bold.

In [12]:
def make_comparison_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    graph_order = ["powerlaw", "er"]

    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": "Erdős-Rényi",
    }

    dataset_order = [
        "small sparse",
        "small dense",
        "large sparse",
        "large dense",
    ]

    lines = [
        r"\begin{table}[p]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{p{2.0cm}p{1.9cm}p{2.2cm}rrr}",
        r"\toprule",
        (
            r"Graphentyp "
            r"& Datensatz "
            r"& Algorithmus "
            r"& \shortstack{Mittlere relative\\Lösungsqualität} "
            r"& Gewinnrate "
            r"& \shortstack{Mittlere\\Laufzeit (s)} \\"
        ),
        r"\midrule",
    ]

    nonempty_graphs = [
        graph_type
        for graph_type in graph_order
        if not df[df["graph_type"] == graph_type].empty
    ]

    for graph_index, graph_type in enumerate(nonempty_graphs):
        graph_df = df[df["graph_type"] == graph_type].copy()

        nonempty_datasets = [
            dataset
            for dataset in dataset_order
            if not graph_df[graph_df["dataset_group"] == dataset].empty
        ]

        graph_row_count = sum(
            len(graph_df[graph_df["dataset_group"] == dataset])
            for dataset in nonempty_datasets
        )

        current_graph_row = 0

        for dataset_index, dataset in enumerate(nonempty_datasets):
            part = graph_df[graph_df["dataset_group"] == dataset].copy()

            part["algorithm"] = pd.Categorical(
                part["algorithm"],
                categories=ALGORITHM_ORDER,
                ordered=True,
            )

            part = part.sort_values("algorithm")

            best_quality = part["mean_relative_to_best"].min()

            for row_index, row in enumerate(part.itertuples(index=False)):
                graph_cell = (
                    rf"\multirow{{{graph_row_count}}}{{*}}{{{graph_labels[graph_type]}}}"
                    if current_graph_row == 0
                    else ""
                )

                dataset_cell = (
                    rf"\multirow{{{len(part)}}}{{*}}{{{dataset}}}"
                    if row_index == 0
                    else ""
                )

                quality = format_number(row.mean_relative_to_best, 6)

                winner_rate = format_percent(row.winner_rate, 1)

                runtime = format_number(row.mean_runtime, 3)

                if np.isclose(row.mean_relative_to_best, best_quality):
                    quality = rf"\textbf{{{quality}}}"

                lines.append(
                    f"{graph_cell} "
                    f"& {dataset_cell} "
                    f"& {latex_algorithm(str(row.algorithm))} "
                    f"& {quality} "
                    f"& {winner_rate} "
                    f"& {runtime} "
                    r"\\"
                )

                current_graph_row += 1

            if dataset_index < len(nonempty_datasets) - 1:
                lines.append(r"\cmidrule(l){2-6}")

        if graph_index < len(nonempty_graphs) - 1:
            lines.append(r"\midrule")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [13]:
comparison_latex = make_comparison_latex_table(
    comparison_summary,
    caption=(
        "Vergleich der mittleren relativen Lösungsqualität, Gewinnrate und mittleren Laufzeit der betrachteten Verfahren. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung."
    ),
    label="tab:algorithm_comparison",
)

print(comparison_latex)

\begin{table}[p]
\centering
\caption{Vergleich der mittleren relativen Lösungsqualität, Gewinnrate und mittleren Laufzeit der betrachteten Verfahren. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung.}
\label{tab:algorithm_comparison}
\begin{tabular}{p{1.8cm}p{2.0cm}p{2.7cm}rrr}
\toprule
Graphentyp & Datensatz & Algorithmus & \shortstack{Mittlere relative\\Lösungsqualität} & Gewinnrate & \shortstack{Mittlere\\Laufzeit (s)} \\
\midrule
\multirow{16}{*}{Powerlaw} & \multirow{4}{*}{small sparse} & \texttt{leiden} & 2.970392 & 0.0\,\% & 0.053 \\
 &  & \texttt{leiden\_mdgp} & 1.095257 & 0.0\,\% & 0.004 \\
 &  & \texttt{kapoce} & 1.049707 & 0.0\,\% & 0.142 \\
 &  & \texttt{mdgp\_plateau} & \textbf{1.000000} & 100.0\,\% & 30.379 \\
\cmidrule(l){2-6}
 & \multirow{4}{*}{small dense} & \texttt{leiden} & 2.538343 & 0.0\,\% & 0.015 \\
 &  & \texttt{lei